# 06 - Station Performance Comparison

This notebook generates a horizontal bar chart comparing on-time performance across all stations.

**Features:**
- Horizontal bars sorted by performance
- Color gradient from best to worst
- Station metadata integration
- Interactive hover with detailed stats

**Output:** `static/plots/station_comparison.html`

In [1]:
import os
import sys
import json
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta

project_root = os.path.abspath(os.path.join(os.getcwd(), '..','..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

DB_PATH = os.path.join(project_root, 'data', 'caltrain_lat_long.db')
GTFS_PATH = os.path.join(project_root, 'gtfs_data')
METADATA_PATH = os.path.join(project_root, 'data', 'station_metadata.json')
OUTPUT_DIR = os.path.join(project_root, 'static', 'plots')

print(f"Database exists: {os.path.exists(DB_PATH)}")
print(f"Metadata exists: {os.path.exists(METADATA_PATH)}")

Database exists: True
Metadata exists: True


## Configuration

In [2]:
# ====================
# CONFIGURATION
# ====================

# Date range
DAYS_TO_ANALYZE = 30  # Last N days, or None for all

# Minimum arrivals to include a station
MIN_ARRIVALS = 50

# Sorting: 'best_first' or 'worst_first' or 'geographic'
SORT_ORDER = 'best_first'

# Color scale for bars
COLOR_SCALE = [
    [0.0, '#EF553B'],   # Red (worst)
    [0.5, '#FECB52'],   # Yellow (mid)
    [1.0, '#00CC96'],   # Green (best)
]

THEME = {
    'bg_color': '#0d1117',
    'text_color': '#c9d1d9',
    'grid_color': '#21262d',
}

FIGURE_WIDTH = 900
FIGURE_HEIGHT = 900  # Taller for all stations

## Load Station Metadata

In [3]:
def load_station_metadata():
    """Load station metadata with ordering information."""
    if os.path.exists(METADATA_PATH):
        with open(METADATA_PATH, 'r') as f:
            data = json.load(f)
        
        stations = {}
        for s in data['stations']:
            for stop_id in s['stop_ids']:
                stations[stop_id] = {
                    'name': s['name'],
                    'id': s['id'],
                    'order': s['order'],
                }
        print(f"Loaded metadata for {len(data['stations'])} stations")
        return stations, data['stations']
    else:
        print("Warning: No metadata file found")
        return {}, []

stop_metadata, station_list = load_station_metadata()

Loaded metadata for 31 stations


## Load and Process Data

In [4]:
sys.path.insert(0, os.path.join(project_root, 'src'))
from utils.time_utils import calculate_time_difference, normalize_time
from utils.geo_utils import haversine

def load_arrivals():
    """Load and process arrival data."""
    conn = sqlite3.connect(DB_PATH)
    
    date_filter = ""
    if DAYS_TO_ANALYZE:
        cutoff = (datetime.now() - timedelta(days=DAYS_TO_ANALYZE)).strftime('%Y-%m-%d')
        date_filter = f" WHERE timestamp >= '{cutoff}'"
    
    query = f"""
    SELECT trip_id, stop_id, vehicle_lat, vehicle_lon, timestamp
    FROM train_locations
    {date_filter}
    """
    raw_df = pd.read_sql_query(query, conn)
    conn.close()
    
    if raw_df.empty:
        return pd.DataFrame()
    
    raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'], errors='coerce')
    raw_df = raw_df.dropna(subset=['timestamp'])
    print(f"Loaded {len(raw_df):,} records")
    
    # Load GTFS
    stops_df = pd.read_csv(os.path.join(GTFS_PATH, 'stops.txt'))
    stops_df = stops_df[stops_df['stop_id'].str.isnumeric()]
    stop_times_df = pd.read_csv(os.path.join(GTFS_PATH, 'stop_times.txt'))
    
    # Type conversion
    raw_df['stop_id'] = pd.to_numeric(raw_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    raw_df['trip_id'] = pd.to_numeric(raw_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    stops_df['stop_id'] = pd.to_numeric(stops_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['stop_id'] = pd.to_numeric(stop_times_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['trip_id'] = pd.to_numeric(stop_times_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    
    raw_df = raw_df.dropna(subset=['trip_id', 'stop_id'])
    
    # Merge
    df = pd.merge(raw_df, stop_times_df[['trip_id', 'stop_id', 'arrival_time']], 
                  on=['trip_id', 'stop_id'], how='inner')
    df = pd.merge(df, stops_df[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], 
                  on='stop_id', how='inner')
    
    df['distance'] = df.apply(lambda r: haversine(
        r['vehicle_lat'], r['vehicle_lon'], r['stop_lat'], r['stop_lon']
    ), axis=1)
    
    df['date'] = df['timestamp'].dt.date
    df['arrival_time'] = df['arrival_time'].apply(normalize_time)
    
    # Arrivals
    min_dist = df.groupby(['trip_id', 'stop_id', 'date'])['distance'].min().reset_index()
    merged = pd.merge(df, min_dist, on=['trip_id', 'stop_id', 'date', 'distance'])
    arrivals = merged.groupby(['trip_id', 'stop_id', 'date']).first().reset_index()
    
    arrivals['delay_min'] = arrivals.apply(
        lambda r: calculate_time_difference(r['arrival_time'], r['timestamp']), axis=1
    )
    arrivals.loc[arrivals.delay_min > 500, 'delay_min'] = 0
    arrivals.loc[arrivals.delay_min < -100, 'delay_min'] = 0
    
    arrivals['on_time'] = arrivals['delay_min'] <= 4
    
    print(f"Processed {len(arrivals):,} arrivals")
    return arrivals

arrivals = load_arrivals()

Base directory: D:\caltrain-tracker
Dotenv path: D:\caltrain-tracker\.env
Dotenv file exists: True
Loaded 302,016 records
Processed 48,518 arrivals


## Calculate Station Statistics

In [5]:
def calculate_station_stats(arrivals):
    """Calculate per-station statistics."""
    if arrivals.empty:
        return pd.DataFrame()
    
    # Map stop_id to parent station name
    def get_station_name(stop_id):
        sid = str(stop_id)
        if sid in stop_metadata:
            return stop_metadata[sid]['name']
        return None
    
    def get_station_order(stop_id):
        sid = str(stop_id)
        if sid in stop_metadata:
            return stop_metadata[sid]['order']
        return 999
    
    arrivals = arrivals.copy()
    arrivals['station_name'] = arrivals['stop_id'].apply(get_station_name)
    arrivals['station_order'] = arrivals['stop_id'].apply(get_station_order)
    
    # Filter to known stations
    arrivals = arrivals[arrivals['station_name'].notna()]
    
    # Group by parent station
    stats = arrivals.groupby(['station_name', 'station_order']).agg(
        on_time_count=('on_time', 'sum'),
        total_count=('on_time', 'count'),
        avg_delay=('delay_min', 'mean'),
        max_delay=('delay_min', 'max'),
    ).reset_index()
    
    stats['on_time_pct'] = (stats['on_time_count'] / stats['total_count'] * 100).round(1)
    stats['avg_delay'] = stats['avg_delay'].round(1)
    
    # Filter by minimum arrivals
    stats = stats[stats['total_count'] >= MIN_ARRIVALS]
    
    print(f"Stats for {len(stats)} stations")
    return stats

station_stats = calculate_station_stats(arrivals)
station_stats.sort_values('on_time_pct', ascending=False).head(10)

Stats for 29 stations


,station_name,station_order,on_time_count,total_count,avg_delay,max_delay,on_time_pct
22,San Jose Diridon,25,1251,1251,-36.5,-0.116667,100.0
21,San Francisco,1,1179,1179,-41.7,0.000000,100.0
2,Belmont,12,1794,1904,0.8,56.950000,94.2
23,San Martin,30,133,142,0.2,11.983333,93.7
17,Redwood City,14,2274,2430,1.0,67.816667,93.6
9,Hayward Park,10,1816,1946,1.2,57.933333,93.3
12,Menlo Park,16,2054,2206,0.9,67.833333,93.1
13,Millbrae,6,2280,2458,1.1,60.916667,92.8
18,San Antonio,19,2048,2207,0.7,68.833333,92.8
5,Burlingame,8,1805,1945,1.1,57.933333,92.8


## Create Horizontal Bar Chart

In [6]:
def create_station_chart(station_stats):
    """Create horizontal bar chart of station performance."""
    if station_stats.empty:
        print("No data to display")
        return None
    
    # Sort based on configuration
    if SORT_ORDER == 'best_first':
        df = station_stats.sort_values('on_time_pct', ascending=True)  # Reversed for horizontal
    elif SORT_ORDER == 'worst_first':
        df = station_stats.sort_values('on_time_pct', ascending=False)
    else:  # geographic
        df = station_stats.sort_values('station_order', ascending=False)
    
    # Normalize for color scale (0-1)
    min_pct = df['on_time_pct'].min()
    max_pct = df['on_time_pct'].max()
    df['color_val'] = (df['on_time_pct'] - min_pct) / (max_pct - min_pct + 0.001)
    
    # Create color array
    colors = px.colors.sample_colorscale(
        [[0, '#EF553B'], [0.5, '#FECB52'], [1, '#00CC96']], 
        df['color_val'].tolist()
    )
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        y=df['station_name'],
        x=df['on_time_pct'],
        orientation='h',
        marker=dict(
            color=colors,
            line=dict(width=0),
        ),
        text=df['on_time_pct'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        textfont=dict(color=THEME['text_color'], size=10),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'On-Time: <b>%{x:.1f}%</b><br>'
            'Arrivals: %{customdata[0]:,}<br>'
            'Avg Delay: %{customdata[1]:.1f} min<br>'
            '<extra></extra>'
        ),
        customdata=df[['total_count', 'avg_delay']].values,
    ))
    
    # Overall average line
    overall_avg = station_stats['on_time_pct'].mean()
    fig.add_vline(
        x=overall_avg,
        line_dash='dash',
        line_color='#58a6ff',
        annotation_text=f'Avg: {overall_avg:.1f}%',
        annotation_position='top',
        annotation_font=dict(color='#58a6ff')
    )
    
    # Summary
    total_arrivals = station_stats['total_count'].sum()
    best_station = station_stats.loc[station_stats['on_time_pct'].idxmax(), 'station_name']
    worst_station = station_stats.loc[station_stats['on_time_pct'].idxmin(), 'station_name']
    
    fig.update_layout(
        title=dict(
            text=f'🚉 Station Performance Comparison<br><span style="font-size:14px;color:{THEME["text_color"]}">Last {DAYS_TO_ANALYZE} days | {int(total_arrivals):,} total arrivals</span>',
            font=dict(size=22, color=THEME['text_color']),
            x=0.5,
            xanchor='center'
        ),
        xaxis=dict(
            title='On-Time Percentage',
            ticksuffix='%',
            tickfont=dict(color=THEME['text_color']),
            # titlefont=dict(color=THEME['text_color']),
            gridcolor=THEME['grid_color'],
            range=[0, 105],
        ),
        yaxis=dict(
            title='',
            tickfont=dict(color=THEME['text_color'], size=11),
        ),
        plot_bgcolor=THEME['bg_color'],
        paper_bgcolor=THEME['bg_color'],
        height=FIGURE_HEIGHT,
        width=FIGURE_WIDTH,
        margin=dict(l=150, r=60, t=100, b=60),
        showlegend=False,
    )
    
    return fig

fig = create_station_chart(station_stats)
if fig:
    fig.show()

## Station Analysis

In [7]:
def analyze_stations(station_stats):
    """Analyze station performance."""
    print("=" * 50)
    print("🚉 STATION PERFORMANCE ANALYSIS")
    print("=" * 50)
    
    # Top performers
    top = station_stats.nlargest(5, 'on_time_pct')
    print("\n✅ Top 5 Stations:")
    for _, row in top.iterrows():
        print(f"  {row['station_name']:25} {row['on_time_pct']:5.1f}% ({int(row['total_count']):,} arrivals)")
    
    # Bottom performers
    bottom = station_stats.nsmallest(5, 'on_time_pct')
    print("\n❌ Bottom 5 Stations:")
    for _, row in bottom.iterrows():
        print(f"  {row['station_name']:25} {row['on_time_pct']:5.1f}% ({int(row['total_count']):,} arrivals)")
    
    # Highest delay
    worst_delay = station_stats.nlargest(5, 'avg_delay')
    print("\n⏱️ Highest Average Delays:")
    for _, row in worst_delay.iterrows():
        print(f"  {row['station_name']:25} {row['avg_delay']:5.1f} min avg")
    
    # Busiest stations
    busiest = station_stats.nlargest(5, 'total_count')
    print("\n🚆 Busiest Stations:")
    for _, row in busiest.iterrows():
        print(f"  {row['station_name']:25} {int(row['total_count']):,} arrivals")

analyze_stations(station_stats)

🚉 STATION PERFORMANCE ANALYSIS

✅ Top 5 Stations:
  San Francisco             100.0% (1,179 arrivals)
  San Jose Diridon          100.0% (1,251 arrivals)
  Belmont                    94.2% (1,904 arrivals)
  San Martin                 93.7% (142 arrivals)
  Redwood City               93.6% (2,430 arrivals)

❌ Bottom 5 Stations:
  Tamien                     82.6% (144 arrivals)
  Capitol                    89.5% (143 arrivals)
  Bayshore                   90.1% (1,945 arrivals)
  22nd Street                90.4% (2,467 arrivals)
  College Park               90.8% (76 arrivals)

⏱️ Highest Average Delays:
  Tamien                      1.9 min avg
  22nd Street                 1.5 min avg
  Bayshore                    1.5 min avg
  Broadway                    1.5 min avg
  Hillsdale                   1.5 min avg

🚆 Busiest Stations:
  South San Francisco       2,468 arrivals
  22nd Street               2,467 arrivals
  Palo Alto                 2,466 arrivals
  Mountain View             2

## Export

In [8]:
if fig:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    output_path = os.path.join(OUTPUT_DIR, 'station_comparison.html')
    fig.write_html(output_path, include_plotlyjs='cdn')
    print(f"✅ Exported to: {output_path}")
    
    # Also export station stats as JSON for the dashboard
    stats_json = station_stats.to_dict(orient='records')
    json_path = os.path.join(project_root, 'static', 'data', 'station_stats.json')
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    with open(json_path, 'w') as f:
        json.dump(stats_json, f, indent=2)
    print(f"✅ Stats JSON: {json_path}")

✅ Exported to: d:\caltrain-tracker\static\plots\station_comparison.html
✅ Stats JSON: d:\caltrain-tracker\static\data\station_stats.json
